# NCBI Entrez Tools

This notebook demonstrates the three NCBI E-utility wrappers provided by proto_tools. The `run_ncbi_esearch` function finds database record IDs matching a query term, `run_ncbi_esummary` retrieves structured metadata for a given record, and `run_ncbi_efetch` downloads full sequences in FASTA format. Together, these three functions support a common bioinformatics workflow: search for records of interest, inspect their metadata, and then fetch the actual sequence data for downstream analysis.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("ncbi")
display_overview("ncbi")
display_docs_section("ncbi", "Background")

# NCBI Entrez

> Three tools wrapping NCBI Entrez E-utilities for searching and retrieving biological sequences (protein, nucleotide, gene):

## Background

**What do these tools access?**
[NCBI](https://www.ncbi.nlm.nih.gov/) (National Center for Biotechnology Information) maintains the world's largest collection of biological sequence databases. The [Entrez](https://www.ncbi.nlm.nih.gov/search/) system provides unified search and retrieval across multiple databases:

* **protein**: Protein sequences from [RefSeq](https://www.ncbi.nlm.nih.gov/refseq/), [GenBank](https://www.ncbi.nlm.nih.gov/genbank/), [Swiss-Prot](https://www.uniprot.org/), [PIR](https://proteininformationresource.org/), and other sources
* **nuccore**: Nucleotide sequences including genomic DNA, mRNA, and other nucleotide records
* **gene**: Gene records with summaries, genomic context, and cross-references

**Why is this important?**

* Gene engineering: retrieve wild-type sequences as starting points for design
* Codon optimization: fetch coding sequences (CDS) to analyze natural codon usage
* Comparative genomics: retrieve sequences from multiple organisms for alignment and analysis
* Regulatory element design: fetch genomic regions upstream/downstream of genes
* Literature-linked sequences: NCBI entries link to [PubMed](https://pubmed.ncbi.nlm.nih.gov/) references and functional annotations

**Scientific foundation:**
NCBI's Entrez [E-utilities](https://www.ncbi.nlm.nih.gov/books/NBK25501/) provide programmatic access to over 40 interconnected databases. The three tools here cover the most common workflow: search for identifiers (ESearch), get metadata (ESummary), and download sequences (EFetch). NCBI search syntax supports field-specific queries (e.g., `[Gene Name]`, `[Organism]`) and Boolean operators for precise retrieval.

## Available tools

In [2]:
display_available_tools("ncbi")

- **`run_ncbi_efetch()`** — Fetch sequences and records from NCBI Entrez by accession or ID
- **`run_ncbi_esearch()`** — Search NCBI Entrez databases by query term to find matching IDs
- **`run_ncbi_esummary()`** — Retrieve record summary metadata from NCBI Entrez by ID

### `run_ncbi_esearch`

The first step in most NCBI workflows is finding the database IDs for records of interest. `run_ncbi_esearch` accepts a free-text query using NCBI's Entrez search syntax, which supports field-qualified terms like gene names and organism filters. It returns a list of matching record IDs from the protein, nuccore, or gene database, which can then be passed directly to `run_ncbi_esummary` or `run_ncbi_efetch` for further retrieval.

In [3]:
from proto_tools import (
    NCBIEsearchInput,
    NCBIFetchConfig,
    run_ncbi_esearch,
)

In [4]:
# Display docs
display_api_reference("ncbi", "input", "run_ncbi_esearch")

# Search for lac repressor protein IDs in Escherichia coli
inputs = NCBIEsearchInput(
    db="protein",
    search_term='"lacI"[Gene] AND "Escherichia coli"[Organism]',
    max_results=5,
)

**Input** — `NCBIEsearchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `db` | `enum` | required | NCBI database to query: 'protein', 'nuccore' (nucleotide core), or 'gene'. Available options: `protein`, `nuccore`, `gene` |
| `search_term` | `string` | required | NCBI search query (e.g. 'lacI\[Gene] AND Escherichia coli\[Organism]'). |
| `max_results` | `integer` | `5` | Maximum number of IDs to return from the search. |

In [5]:
# Display config docs
display_api_reference("ncbi", "config", "run_ncbi_esearch")

# Config is shared across all three NCBI functions | see docs above for all fields
config = NCBIFetchConfig()

**Config** — `NCBIFetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `ncbi_api_key` | `string` |  | Optional NCBI API key for higher rate limits. |
| `ncbi_email` | `string` |  | Optional contact email for NCBI requests. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the search
search_result = run_ncbi_esearch(inputs, config)

In [7]:
# Display output docs
display_api_reference("ncbi", "output", "run_ncbi_esearch")

# Inspect the returned IDs
print(f"IDs found: {search_result.ids}")

**Output** — `NCBIEsearchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `ids` | `List[string]` |  | List of NCBI IDs matching the search query. |

IDs found: ['3259760730', '3259521955', '3259417777', '2906245810', '2495843448']


### `run_ncbi_esummary`

Once you have record IDs from a search, `run_ncbi_esummary` retrieves structured metadata without downloading the full record. This is useful for inspecting gene annotations, genomic coordinates, organism information, and other catalog-level details. It accepts a database name and a single accession or NCBI ID, and returns the full summary document as a parsed dictionary.

In [8]:
from proto_tools import (
    NCBIEsummaryInput,
    run_ncbi_esummary,
)

In [9]:
# Display docs
display_api_reference("ncbi", "input", "run_ncbi_esummary")

# Fetch the gene summary for human TP53 using its NCBI Gene ID
esummary_inputs = NCBIEsummaryInput(
    db="gene",
    identifier="7157",
)

**Input** — `NCBIEsummaryInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `db` | `enum` | required | NCBI database to query: 'protein', 'nuccore' (nucleotide core), or 'gene'. Available options: `protein`, `nuccore`, `gene` |
| `identifier` | `string` | required | Accession or NCBI ID to summarize (e.g. 'NP\_000537.3', '7157'). |

In [10]:
# Display config docs
display_api_reference("ncbi", "config", "run_ncbi_esummary")

# Config is shared across all three NCBI functions | see docs above for all fields
esummary_config = NCBIFetchConfig()

**Config** — `NCBIFetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `ncbi_api_key` | `string` |  | Optional NCBI API key for higher rate limits. |
| `ncbi_email` | `string` |  | Optional contact email for NCBI requests. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [11]:
# Run the summary fetch
summary_result = run_ncbi_esummary(esummary_inputs, esummary_config)

In [12]:
# Display output docs
display_api_reference("ncbi", "output", "run_ncbi_esummary")

# Inspect gene metadata for TP53
gene_id = esummary_inputs.identifier
gene_data = summary_result.summary.get(gene_id, {})
print(f"Name: {gene_data.get('name')}")
print(f"Description: {gene_data.get('description')}")
print(f"Organism: {gene_data.get('organism', {}).get('scientificname')}")

genomic_info = gene_data.get("genomicinfo", [])
if genomic_info:
    region = genomic_info[0]
    print(f"Chromosome: {region.get('chraccver')}")
    print(f"Coordinates: {region.get('chrstart')} to {region.get('chrstop')}")

**Output** — `NCBIEsummaryOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `summary` | `Dict[string, any]` |  | Record summary data returned by esummary. |
| `source_url` | `string` | required | Sanitized URL used for the request. |

Name: TP53
Description: tumor protein p53
Organism: Homo sapiens
Chromosome: NC_000017.11
Coordinates: 7687489 to 7668420


### `run_ncbi_efetch`

`run_ncbi_efetch` downloads full sequence records from NCBI in FASTA format. It is the workhorse for retrieving protein or nucleotide sequences needed for alignment, structure prediction, or other downstream analyses. In addition to full-record retrieval, it supports subsequence extraction by specifying start and stop coordinates (1-indexed, inclusive), and strand selection for nucleotide records, making it suitable for pulling specific gene loci or regulatory elements from large chromosome assemblies.

In [13]:
from proto_tools import (
    NCBIEfetchInput,
    run_ncbi_efetch,
)

In [14]:
# Display docs
display_api_reference("ncbi", "input", "run_ncbi_efetch")

# Fetch the TP53 protein sequence by RefSeq accession
efetch_inputs = NCBIEfetchInput(
    db="protein",
    identifier="NP_000537.3",
    return_format="fasta",
)

**Input** — `NCBIEfetchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `db` | `enum` | required | NCBI database to query: 'protein', 'nuccore' (nucleotide core), or 'gene'. Available options: `protein`, `nuccore`, `gene` |
| `identifier` | `string` | required | Accession or NCBI ID to fetch (e.g. 'NP\_000537.3'). |
| `return_format` | `enum` | `fasta` | NCBI rettype: 'fasta' for sequences or 'fasta\_cds\_na' for coding DNA sequences. Available options: `fasta`, `fasta_cds_na` |
| `seq_start` | `integer` |  | Start position for subsequence extraction (1-indexed, inclusive). |
| `seq_stop` | `integer` |  | Stop position for subsequence extraction (1-indexed, inclusive). |
| `strand` | `string` |  | Strand for nucleotide retrieval (+ or -). |

In [15]:
# Display config docs
display_api_reference("ncbi", "config", "run_ncbi_efetch")

# Config is shared across all three NCBI functions | see docs above for all fields
efetch_config = NCBIFetchConfig()

**Config** — `NCBIFetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `ncbi_api_key` | `string` |  | Optional NCBI API key for higher rate limits. |
| `ncbi_email` | `string` |  | Optional contact email for NCBI requests. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [16]:
# Run the fetch
fetch_result = run_ncbi_efetch(efetch_inputs, efetch_config)

In [17]:
# Display output docs
display_api_reference("ncbi", "output", "run_ncbi_efetch")

# Inspect the fetched FASTA record
record = fetch_result.fasta_records[0]
print(f"Accession: {record.accession}")
print(f"Header: {record.header[:80]}...")
print(f"Sequence length: {len(record.sequence)}")
print(f"Preview: {record.sequence[:60]}...")

**Output** — `NCBIEfetchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `fasta_records` | `List[NCBIFastaRecord]` |  | Parsed FASTA records from efetch. |
| `header` | `string` | required | FASTA header line (without >). |
| `sequence` | `string` | required | Sequence string with whitespace stripped. |
| `accession` | `string` |  | Best-effort accession extracted from header. |
| `source_url` | `string` | required | Sanitized URL used for the request. |

Accession: NP_000537.3
Header: NP_000537.3 cellular tumor antigen p53 isoform a [Homo sapiens]...
Sequence length: 393
Preview: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...


#### Genomic Subsequence Extraction

When working with large genomes, you often need just a small region rather than the entire chromosome. The efetch function supports coordinate-based extraction with strand selection, which is useful for pulling promoter regions, specific gene loci, or regulatory elements. Here we retrieve a 200 bp region from the lacI locus on the minus strand of the *E. coli* K-12 genome.

In [18]:
# Fetch a 200 bp region from the lacI locus (minus strand)
subseq_inputs = NCBIEfetchInput(
    db="nuccore",
    identifier="NC_000913.3",
    return_format="fasta",
    seq_start=366428,
    seq_stop=366628,
    strand="-",
)

subseq_result = run_ncbi_efetch(subseq_inputs, efetch_config)

record = subseq_result.fasta_records[0]
print(f"Accession: {record.accession}")
print(f"Sequence length: {len(record.sequence)}")
print(f"Preview: {record.sequence[:80]}...")
print(f"Source URL: {subseq_result.source_url}")

Accession: NC_000913.3:c366628-366428
Sequence length: 201
Preview: CTGCTGGGGCAAACCAGCGTGGACCGCTTGCTGCAACTCTCTCAGGGCCAGGCGGTGAAGGGCAATCAGCTGTTGCCCGT...
Source URL: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_000913.3&rettype=fasta&retmode=text&seq_start=366428&seq_stop=366628&strand=2&tool=proto_tools_ncbi_fetch


## Export Results

Results from all three functions can be exported to JSON format for downstream processing or archival. This is useful for building reproducible pipelines where intermediate results need to be saved between steps.

In [19]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

fetch_result.export("ncbi_efetch_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'ncbi_efetch_results.json'}")

summary_result.export("ncbi_gene_summary", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'ncbi_gene_summary.json'}")

Exported to example_output/ncbi_efetch_results.json
Exported to example_output/ncbi_gene_summary.json
